# W06 Extension v2 — Upgrading Current Models Against LambdaRank

This notebook evaluates second-generation feature engineering for the current model families and tests whether any treatment beats the existing engineered LambdaRank benchmark at the primary operational cutoff, P@50.

> Scope: cross-client, current-snapshot ranking on the 30k starter data. The result is not future forecasting evidence.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'skills' / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')

ROOT = find_repo_root(Path.cwd())
SCRIPT = ROOT / 'work' / 'scripts' / 'feature_engineering_v2_experiment.py'
RESULTS = ROOT / 'work' / 'outputs' / 'feature_engineering_v2_results.json'
OOF = ROOT / 'work' / 'outputs' / 'feature_engineering_v2_oof.csv'
FEATURES = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT or not RESULTS.exists() or not OOF.exists():
    completed = subprocess.run([sys.executable, '-X', 'utf8', str(SCRIPT)], cwd=ROOT, check=True, capture_output=True, text=True)
    print(completed.stdout)
else:
    assert RESULTS.stat().st_mtime >= SCRIPT.stat().st_mtime, 'Results are older than the experiment script; rerun required.'
    print('Using current full-run receipt and OOF predictions; set RUN_FULL_EXPERIMENT=True to retrain all candidates.')

payload = json.loads(RESULTS.read_text(encoding='utf-8'))
oof = pd.read_csv(OOF)
features = pd.read_csv(FEATURES)
assert len(oof) == payload['rows'] == 30000
print(f"Rows={payload['rows']:,}; clients={payload['clients']}; matrix={tuple(payload['matrix_shape'])}")
print(f"Winner={payload['winner']}; improvement={payload['improvement_pp']:+.1f}pp")

Using current full-run receipt and OOF predictions; set RUN_FULL_EXPERIMENT=True to retrain all candidates.
Rows=30,000; clients=32; matrix=(30000, 324)
Winner=F7_lambda_pair_equal_blend; improvement=+0.8pp


## FE v2 design

The upgrade adds predictor-only feature families:

- cross-channel intensity and residual features, such as pageviews per session and search-versus-analytics gaps;
- log, ratio, age/staleness, position, visibility, and demand interactions;
- robust within-client and within-content-type percentiles and IQR-scaled deviations;
- predictor-only quantile buckets and categorical crosses;
- fold-safe leave-one-out target encoding, calculated from training labels only.

The script blocks trend sources, latest/previous 30-day windows, IDs, and product scores from model features. The previous-window family rejected in v1 remains excluded.

In [2]:
comparison = pd.DataFrame(payload['comparison']).sort_values('mean_p_at_50', ascending=False)
shown = comparison[[
    'candidate', 'mean_p_at_20', 'mean_p_at_50', 'std_p_at_50',
    'mean_p_at_100', 'mean_roc_auc', 'mean_average_precision'
]].copy()
for column in ['mean_p_at_20', 'mean_p_at_50', 'std_p_at_50', 'mean_p_at_100']:
    shown[column] = (shown[column] * 100).round(1).astype(str) + '%'
shown['mean_roc_auc'] = shown['mean_roc_auc'].round(4)
shown['mean_average_precision'] = shown['mean_average_precision'].round(4)
print(shown.to_string(index=False))

assert payload['winner_mean_p_at_50'] > payload['benchmark_mean_p_at_50']
print(f"\nBenchmark P@50: {payload['benchmark_mean_p_at_50']:.1%}")
print(f"Winner P@50:    {payload['winner_mean_p_at_50']:.1%}")
print(f"Observed gain:  {payload['improvement_pp']:+.1f}pp")

                        candidate mean_p_at_20 mean_p_at_50 std_p_at_50 mean_p_at_100  mean_roc_auc  mean_average_precision
       F7_lambda_pair_equal_blend        92.0%        92.4%        3.3%         88.6%        0.6431                  0.6840
         B0_engineered_lambdarank        94.0%        91.6%        4.6%         85.0%        0.6280                  0.6703
               B1_base_lambdarank        90.0%        90.8%        2.7%         90.4%        0.6467                  0.6846
F9_lambda_fev2_ranker_equal_blend        90.0%        88.8%        5.2%         84.8%        0.6262                  0.6657
              F5_fev2_extra_trees        88.0%        88.4%        6.5%         84.8%        0.6735                  0.6863
           F6_fev2_lambdarank_k50        90.0%        86.8%        8.6%         82.2%        0.6126                  0.6524
                  F4_fev2_xgboost        86.0%        84.4%        1.7%         81.6%        0.6740                  0.6882
        

In [3]:
paired = pd.DataFrame(payload['paired_differences'])
print(paired.to_string(index=False, formatters={
    'winner_p_at_50': '{:.1%}'.format,
    'benchmark_p_at_50': '{:.1%}'.format,
    'improvement_pp': '{:+.1f}'.format,
}))
print(f"Winner improved P@50 in {(paired['improvement_pp'] > 0).sum()} of {len(paired)} folds.")
print(f"Fold-level improvement range: {paired['improvement_pp'].min():+.1f}pp to {paired['improvement_pp'].max():+.1f}pp")

 fold winner_p_at_50 benchmark_p_at_50 improvement_pp
    1          88.0%             86.0%           +2.0
    2          94.0%             90.0%           +4.0
    3          94.0%             90.0%           +4.0
    4          90.0%             94.0%           -4.0
    5          96.0%             98.0%           -2.0
Winner improved P@50 in 3 of 5 folds.
Fold-level improvement range: -4.0pp to +4.0pp


In [4]:
def precision_at_k(y, score, k):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    cutoff = min(k, len(y))
    return float(y[np.argsort(-score, kind='mergesort')[:cutoff]].mean())

benchmark_col = payload['benchmark']
winner_col = payload['winner']
client_rows = []
for _, client_frame in oof.groupby('client_id'):
    n = len(client_frame)
    if n < 50:
        continue
    row = {
        'n': n,
        'base_rate': client_frame['is_declining_label'].mean(),
        'benchmark_p50': precision_at_k(client_frame['is_declining_label'], client_frame[benchmark_col], 50),
        'winner_p50': precision_at_k(client_frame['is_declining_label'], client_frame[winner_col], 50),
    }
    if client_frame['is_declining_label'].nunique() == 2:
        row['benchmark_auc'] = roc_auc_score(client_frame['is_declining_label'], client_frame[benchmark_col])
        row['winner_auc'] = roc_auc_score(client_frame['is_declining_label'], client_frame[winner_col])
    client_rows.append(row)
client_stress = pd.DataFrame(client_rows)
client_stress['improvement_pp'] = 100 * (client_stress['winner_p50'] - client_stress['benchmark_p50'])
summary = {
    'eligible_clients': len(client_stress),
    'macro_benchmark_p50': client_stress['benchmark_p50'].mean(),
    'macro_winner_p50': client_stress['winner_p50'].mean(),
    'clients_improved': int((client_stress['improvement_pp'] > 0).sum()),
    'clients_tied': int((client_stress['improvement_pp'] == 0).sum()),
    'clients_worsened': int((client_stress['improvement_pp'] < 0).sum()),
}
print(pd.Series(summary).to_string())
print(f"Macro-client P@50 change: {100 * (summary['macro_winner_p50'] - summary['macro_benchmark_p50']):+.1f}pp")
print('No client identifier is displayed.')

eligible_clients       25.0000
macro_benchmark_p50     0.7048
macro_winner_p50        0.7248
clients_improved       10.0000
clients_tied            9.0000
clients_worsened        6.0000
Macro-client P@50 change: +2.0pp
No client identifier is displayed.


In [5]:
print('Strongest pure FE current model: ExtraTrees')
extra_importance = pd.DataFrame(payload['top_feature_importance']['F5_fev2_extra_trees']).head(15)
print(extra_importance.to_string(index=False, formatters={'mean_importance': '{:.4f}'.format}))
print('\nTop three ExtraTrees features:', ', '.join(extra_importance['feature'].head(3)))
print('The final winner is a rank ensemble; its components use the base and first-generation engineered feature sets.')

Strongest pure FE current model: ExtraTrees
                  feature mean_importance
      visibility_decile_0          0.1272
           active_day_gap          0.0291
            age_tier_365+          0.0282
  impression_day_coverage          0.0241
    days_with_impressions          0.0240
         content_age_days          0.0173
     log_content_age_days          0.0151
 client_clicks_percentile          0.0115
client2_clicks_percentile          0.0112
         active_day_ratio          0.0109
      scrolls_per_session          0.0099
 type_position_percentile          0.0099
             log_position          0.0097
      type_ctr_percentile          0.0097
              scroll_rate          0.0094

Top three ExtraTrees features: visibility_decile_0, active_day_gap, age_tier_365+
The final winner is a rank ensemble; its components use the base and first-generation engineered feature sets.


In [6]:
queue_rows = []
winner_top_indices = []
for fold, fold_frame in oof.groupby('fold'):
    benchmark_top = fold_frame.nlargest(50, benchmark_col)
    winner_top = fold_frame.nlargest(50, winner_col)
    winner_top_indices.extend(winner_top.index.tolist())
    queue_rows.append({
        'fold': int(fold),
        'benchmark_true_positives': int(benchmark_top['is_declining_label'].sum()),
        'winner_true_positives': int(winner_top['is_declining_label'].sum()),
        'difference': int(winner_top['is_declining_label'].sum() - benchmark_top['is_declining_label'].sum()),
    })
queue_audit = pd.DataFrame(queue_rows)
print(queue_audit.to_string(index=False))
print(f"Across five top-50 queues, the ensemble retrieves {queue_audit['difference'].sum():+d} additional positives.")

false_positive_indices = [i for i in winner_top_indices if int(oof.loc[i, 'is_declining_label']) == 0]
false_positive_indices = sorted(false_positive_indices, key=lambda i: oof.loc[i, winner_col], reverse=True)[:3]
case_columns = ['content_type', 'impression_tier', 'position_tier', 'freshness_tier', 'avg_position', 'days_since_last_update']
cases = features.loc[false_positive_indices, case_columns].copy().reset_index(drop=True)
cases.insert(0, 'case', [f'FP case {i + 1}' for i in range(len(cases))])
cases['why_hard'] = 'High ensemble rank but the threshold label is not down; inspect seasonality, volatility, and editorial context.'
print(cases.to_string(index=False))
print('Cases are anonymous: no client name, URL, query, or identifier is displayed.')

 fold  benchmark_true_positives  winner_true_positives  difference
    1                        43                     44           1
    2                        45                     47           2
    3                        45                     47           2
    4                        47                     45          -2
    5                        49                     48          -1
Across five top-50 queues, the ensemble retrieves +2 additional positives.
     case    content_type impression_tier position_tier freshness_tier  avg_position  days_since_last_update                                                                                                        why_hard
FP case 1 keyword article        moderate        page_1         91-180           4.7                     104 High ensemble rank but the threshold label is not down; inspect seasonality, volatility, and editorial context.
FP case 2 keyword article        moderate        page_1         91-180           

## Honest conclusion

The FE v2 treatments upgraded multiple current model families, but no pure classifier surpassed engineered LambdaRank: ExtraTrees was strongest at 88.4% mean P@50. Fold-safe target encoding transferred poorly and was rejected.

The successful upgrade is an equal fold-wise rank ensemble of base-feature and engineered LambdaRank. It achieved **92.4% ± 3.3% P@50**, compared with **91.6% ± 4.6%** for engineered LambdaRank: an observed **+0.8pp** gain. It also improved P@100, ROC-AUC, and Average Precision, but P@20 decreased from 94% to 92%.

This is a marginal development-fold improvement—two additional positives across 250 reviewed slots—not evidence of a large universal gain. The result should remain a candidate until tested on an untouched temporal/client evaluation.

## Self-check

- [x] FE v2 and benchmarks use the exact same five client folds.
- [x] Trend sources, 30-day target siblings, IDs, and product scores are excluded.
- [x] Supervised target encodings are calculated using training labels only.
- [x] Every pure current-model upgrade is reported, including failures.
- [x] P@20/P@50/P@100, fold dispersion, per-client stress, importance, and errors are visible.
- [x] Notebook runs top to bottom with fixed seeds and current receipts.